# W3 Day4 (04/24 周四): LLM架构深入

---

## 学习目标

- 理解 **MHA / MQA / GQA** 三种注意力变体的原理、实现与显存差异
- 掌握 **RoPE**（旋转位置编码）和 **ALiBi**（线性注意力偏置）的设计思想
- 拆解 **LLaMA** 的核心组件：Pre-RMSNorm、SwiGLU、RoPE、GQA
- 理解 **Scaling Laws** 与 Chinchilla 最优训练策略

## 面试高频问题（8题）

| # | 问题 |
|---|------|
| 1 | MHA、MQA、GQA 的区别是什么？各自的显存占用如何计算？ |
| 2 | 为什么 MQA 能显著减少 KV cache 的显存占用？ |
| 3 | RoPE 的旋转矩阵是如何构造的？为什么它比学习式位置编码更好？ |
| 4 | ALiBi 和 RoPE 的本质区别是什么？各自适用什么场景？ |
| 5 | LLaMA 相比 GPT 做了哪些架构改进？每项改进的作用是什么？ |
| 6 | SwiGLU 激活函数相比 ReLU/GELU 有什么优势？ |
| 7 | Chinchilla Scaling Law 的核心结论是什么？对训练有什么指导意义？ |
| 8 | 如何计算一个 7B 模型在 4096 序列长度下的 KV cache 显存？ |

## 目录

1. **MHA 回顾** — Q/K/V 计算与缩放点积注意力
2. **Multi-Head Attention 实现** — 从零搭建 MHA
3. **MQA 理论与实现** — 共享 K/V 头
4. **GQA 理论与实现** — 分组 K/V 头
5. **KV Cache 显存分析** — 公式推导与计算
6. **RoPE 理论与实现** — 旋转位置编码
7. **ALiBi 理论与实现** — 线性注意力偏置
8. **LLaMA 组件拆解** — Pre-RMSNorm、SwiGLU、GQA
9. **Scaling Laws** — Chinchilla 最优训练
10. **面试 Q&A 总结**

## 1. MHA 回顾 — Q/K/V 与缩放点积注意力

### 核心公式

给定输入序列 $X \in \mathbb{R}^{L \times d_{model}}$：

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

其中 $W^Q, W^K \in \mathbb{R}^{d_{model} \times d_k}$，$W^V \in \mathbb{R}^{d_{model} \times d_v}$

**缩放点积注意力（Scaled Dot-Product Attention）：**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**多头注意力（Multi-Head Attention）：**

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

$$\text{head}_i = \text{Attention}(XW^Q_i, XW^K_i, XW^V_i)$$

### MHA 的 KV Cache 问题

在自回归推理中，每层需要缓存所有 $h$ 个头的 K 和 V：

$$\text{KV cache/层} = 2 \times h \times d_k \times L \times \text{dtype\_size}$$

当 $h$ 很大时（如 LLaMA-70B 有 64 头），KV cache 成为推理瓶颈。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 配色方案
C_RED    = '#c2553a'
C_GREEN  = '#5a6b4a'
C_DARK   = '#2d2a26'
C_ORANGE = '#e07b39'
C_BLUE   = '#3a7ca5'
C_GRAY   = '#888888'

print('环境准备完成')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# Multi-Head Attention (MHA) 从零实现
# ============================================================

class MultiHeadAttention(nn.Module):
    """
    标准多头注意力 (MHA)
    每个头都有独立的 Q, K, V 投影
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model 必须能被 n_heads 整除'
        self.d_model  = d_model
        self.n_heads  = n_heads   # 头数 h
        self.head_dim = d_model // n_heads  # 每个头的维度 d_k
        
        # Q, K, V 投影矩阵 —— 每个头独立
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # h 个 Q 头
        self.W_k = nn.Linear(d_model, d_model, bias=False)  # h 个 K 头
        self.W_v = nn.Linear(d_model, d_model, bias=False)  # h 个 V 头
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # 输出投影
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, d_model)
        返回: (batch, seq_len, d_model)
        """
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim
        
        # Step 1: 线性投影 -> (B, L, D)
        Q = self.W_q(x)  # (B, L, D)
        K = self.W_k(x)  # (B, L, D)
        V = self.W_v(x)  # (B, L, D)
        
        # Step 2: 拆分成多头 -> (B, H, L, d)
        Q = Q.view(B, L, H, d).transpose(1, 2)  # (B, H, L, d)
        K = K.view(B, L, H, d).transpose(1, 2)  # (B, H, L, d)
        V = V.view(B, L, H, d).transpose(1, 2)  # (B, H, L, d)
        
        # Step 3: 缩放点积注意力
        # scores = Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d)
        # scores: (B, H, L, L)
        
        # 因果掩码（用于 decoder）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)  # (B, H, L, L)
        attn = self.dropout(attn)
        
        # Step 4: 加权求和
        out = torch.matmul(attn, V)  # (B, H, L, d)
        
        # Step 5: 拼接所有头 -> (B, L, D)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        
        # Step 6: 输出投影
        out = self.W_o(out)
        return out, attn


# ---- 测试 MHA ----
try:
    B, L, D, H = 2, 8, 64, 8  # batch=2, seq_len=8, d_model=64, n_heads=8
    mha = MultiHeadAttention(d_model=D, n_heads=H)
    x = torch.randn(B, L, D)
    
    # 因果掩码
    causal_mask = torch.tril(torch.ones(L, L)).unsqueeze(0).unsqueeze(0)  # (1, 1, L, L)
    
    out, attn_weights = mha(x, mask=causal_mask)
    
    print(f'输入形状:  {x.shape}')
    print(f'输出形状:  {out.shape}')
    print(f'注意力权重: {attn_weights.shape}')
    print(f'参数量:    {sum(p.numel() for p in mha.parameters()):,}')
    
    # 验证因果掩码：上三角应为 0
    print(f'\n因果掩码验证（第0头，第0个batch）：')
    print(attn_weights[0, 0].detach().numpy().round(3))
except Exception as e:
    print(f'错误: {e}')

## 2. MQA — Multi-Query Attention

### 核心思想

**所有头共享同一组 K 和 V**，只有 Q 保持多头。

MQA 由 Noam Shazeer 在 2019 年提出（"Fast Transformer Decoding"）：

$$Q_i = XW^Q_i \quad (i = 1, \ldots, h)$$

$$K = XW^K, \quad V = XW^V \quad (\text{所有头共享})$$

$$\text{head}_i = \text{Attention}(Q_i, K, V)$$

### 显存对比

| 指标 | MHA | MQA |
|------|-----|-----|
| K 投影参数 | $h \times d_{model} \times d_k$ | $d_{model} \times d_k$ |
| V 投影参数 | $h \times d_{model} \times d_v$ | $d_{model} \times d_v$ |
| **KV cache 缩减** | 基准 | **$1/h$** |
| 质量影响 | 基准 | 轻微下降 |

**实际收益**：对于 70B 模型（$h=64$），MQA 将 KV cache 缩减到 MHA 的 **1/64**。

### 使用 MQA 的模型

- PaLM（Google）、Falcon、StarCoder

In [ ]:
# ============================================================
# Multi-Query Attention (MQA) 实现
# 核心区别：K, V 只有 1 个头，通过 expand 广播到所有 Q 头
# ============================================================

class MultiQueryAttention(nn.Module):
    """
    Multi-Query Attention (MQA)
    Q: h 个头（与 MHA 相同）
    K, V: 仅 1 个头，共享给所有 Q 头
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        
        # Q 仍然有 h 个头
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        # K, V 只有 1 个头！这是 MQA 的关键
        self.W_k = nn.Linear(d_model, self.head_dim, bias=False)  # 只输出 d_k
        self.W_v = nn.Linear(d_model, self.head_dim, bias=False)  # 只输出 d_k
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim
        
        # Q: (B, H, L, d) — 多头
        Q = self.W_q(x).view(B, L, H, d).transpose(1, 2)
        
        # K, V: (B, 1, L, d) — 单头，用 expand 广播
        K = self.W_k(x).view(B, L, 1, d).transpose(1, 2)  # (B, 1, L, d)
        V = self.W_v(x).view(B, L, 1, d).transpose(1, 2)  # (B, 1, L, d)
        
        # expand 广播：(B, 1, L, d) -> (B, H, L, d)
        # 注意：expand 不分配新内存，零拷贝
        K = K.expand(B, H, L, d)
        V = V.expand(B, H, L, d)
        
        # 注意力计算（与 MHA 完全相同）
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        out = self.W_o(out)
        return out, attn


# ---- 测试 MQA 并与 MHA 对比 ----
try:
    mqa = MultiQueryAttention(d_model=D, n_heads=H)
    out_mqa, attn_mqa = mqa(x, mask=causal_mask)
    
    mha_params = sum(p.numel() for p in mha.parameters())
    mqa_params = sum(p.numel() for p in mqa.parameters())
    
    print(f'MQA 输出形状: {out_mqa.shape}')
    print(f'\n=== 参数量对比 ===')
    print(f'MHA 参数量: {mha_params:,}')
    print(f'MQA 参数量: {mqa_params:,}')
    print(f'参数缩减比: {mha_params / mqa_params:.2f}x')
    
    # KV cache 大小对比
    # MHA: 2 * n_heads * head_dim * seq_len
    # MQA: 2 * 1 * head_dim * seq_len
    kv_mha = 2 * H * d * L
    kv_mqa = 2 * 1 * d * L
    print(f'\n=== KV Cache 对比 (单层, seq_len={L}) ===')
    print(f'MHA KV cache: {kv_mha} 元素')
    print(f'MQA KV cache: {kv_mqa} 元素')
    print(f'KV cache 缩减: {kv_mha / kv_mqa:.0f}x')
except Exception as e:
    print(f'错误: {e}')

## 3. GQA — Grouped-Query Attention

### 核心思想

GQA 是 MHA 和 MQA 的**折中方案**：将 $h$ 个 Q 头分成 $g$ 组，每组共享一组 K/V。

- 当 $g = h$ 时，GQA 退化为 **MHA**（每个 Q 头有自己的 K/V）
- 当 $g = 1$ 时，GQA 退化为 **MQA**（所有 Q 头共享一组 K/V）
- 一般取 $g = h / n$，例如 LLaMA-70B：$h=64, g=8$（每 8 个 Q 头共享一组 K/V）

$$\text{head}_i = \text{Attention}(Q_i, K_{g(i)}, V_{g(i)})$$

其中 $g(i) = \lfloor i \cdot n_{kv} / h \rfloor$ 是第 $i$ 个 Q 头对应的 KV 组索引。

### GQA 的优势

| 特性 | MHA | GQA | MQA |
|------|-----|-----|-----|
| KV 头数 | $h$ | $g$ ($1 < g < h$) | $1$ |
| KV cache | 基准 | $g/h$ | $1/h$ |
| 质量 | 最好 | 接近 MHA | 略差 |
| 推理速度 | 基准 | 较快 | 最快 |

**LLaMA 2/3 选择 GQA**：在质量和效率之间取得最佳平衡。

| 模型 | Q heads ($h$) | KV heads ($g$) | 分组比 |
|------|:---:|:---:|:---:|
| LLaMA-7B | 32 | 32 | 1:1 (MHA) |
| LLaMA-2 7B | 32 | 32 | 1:1 (MHA) |
| LLaMA-2 70B | 64 | 8 | 8:1 (GQA) |
| LLaMA-3 8B | 32 | 8 | 4:1 (GQA) |
| LLaMA-3 70B | 64 | 8 | 8:1 (GQA) |

In [ ]:
# ============================================================
# Grouped-Query Attention (GQA) 实现
# 关键参数: n_kv_heads 控制 K/V 的头数
# ============================================================

class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (GQA)
    n_heads:    Q 的头数 (h)
    n_kv_heads: K/V 的头数 (g), 须满足 n_heads % n_kv_heads == 0
    每个 KV 头被 n_heads // n_kv_heads 个 Q 头共享
    """
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model 必须能被 n_heads 整除'
        assert n_heads % n_kv_heads == 0, 'n_heads 必须能被 n_kv_heads 整除'
        
        self.d_model   = d_model
        self.n_heads   = n_heads      # Q 头数
        self.n_kv_heads = n_kv_heads  # K/V 头数
        self.head_dim  = d_model // n_heads
        self.n_rep     = n_heads // n_kv_heads  # 每个 KV 头被几个 Q 头共享
        
        # Q 有 n_heads 个头
        self.W_q = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        # K, V 只有 n_kv_heads 个头
        self.W_k = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        将 K/V 从 (B, n_kv_heads, L, d) 扩展到 (B, n_heads, L, d)
        方法: 在 n_kv_heads 维度上 repeat n_rep 次
        """
        if self.n_rep == 1:
            return x  # n_kv_heads == n_heads 时无需重复
        B, n_kv, L, d = x.shape
        # (B, n_kv, L, d) -> (B, n_kv, 1, L, d) -> (B, n_kv, n_rep, L, d) -> (B, n_heads, L, d)
        return x[:, :, None, :, :].expand(B, n_kv, self.n_rep, L, d).reshape(B, self.n_heads, L, d)
    
    def forward(self, x, mask=None):
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim
        
        # Q: (B, H, L, d)
        Q = self.W_q(x).view(B, L, H, d).transpose(1, 2)
        # K, V: (B, n_kv_heads, L, d)
        K = self.W_k(x).view(B, L, self.n_kv_heads, d).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_kv_heads, d).transpose(1, 2)
        
        # 将 KV 扩展到与 Q 相同的头数
        K = self._repeat_kv(K)  # (B, H, L, d)
        V = self._repeat_kv(V)  # (B, H, L, d)
        
        # 注意力计算
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        out = self.W_o(out)
        return out, attn


# ---- 测试 GQA 并与 MHA/MQA 对比 ----
try:
    # GQA: 8 个 Q 头, 2 个 KV 头 (4:1 分组)
    gqa = GroupedQueryAttention(d_model=D, n_heads=H, n_kv_heads=2)
    out_gqa, attn_gqa = gqa(x, mask=causal_mask)
    
    gqa_params = sum(p.numel() for p in gqa.parameters())
    
    print(f'GQA 输出形状: {out_gqa.shape}')
    print(f'GQA n_rep (每个KV头被几个Q头共享): {gqa.n_rep}')
    
    print(f'\n=== 三种注意力变体参数量对比 ===')
    print(f'MHA ({H}Q/{H}KV): {mha_params:>8,} 参数')
    print(f'GQA ({H}Q/2KV): {gqa_params:>8,} 参数')
    print(f'MQA ({H}Q/1KV): {mqa_params:>8,} 参数')
    
    # KV cache 对比
    kv_gqa = 2 * 2 * d * L  # 2 个 KV 头
    print(f'\n=== KV Cache 对比 (单层, seq_len={L}) ===')
    print(f'MHA: {kv_mha:>6} 元素')
    print(f'GQA: {kv_gqa:>6} 元素')
    print(f'MQA: {kv_mqa:>6} 元素')
except Exception as e:
    print(f'错误: {e}')

## 4. KV Cache 显存分析

### KV Cache 公式

自回归推理时，每层需要缓存所有 token 的 K 和 V 向量：

$$\text{KV Cache} = 2 \times n_{layers} \times n_{kv\_heads} \times d_{head} \times seq\_len \times batch \times \text{dtype\_size}$$

其中：
- **2**：K 和 V 各一份
- $n_{layers}$：Transformer 层数
- $n_{kv\_heads}$：KV 头数（MHA=$h$, GQA=$g$, MQA=$1$）
- $d_{head}$：每个头的维度（通常 128）
- $seq\_len$：序列长度
- $batch$：批大小
- $\text{dtype\_size}$：数据类型字节数（FP16=2, FP32=4）

### 典型配置对比

| 模型 | 参数量 | 层数 | $n_{kv\_heads}$ | $d_{head}$ | 类型 |
|------|:---:|:---:|:---:|:---:|:---:|
| LLaMA-7B | 7B | 32 | 32 | 128 | MHA |
| LLaMA-2 7B | 7B | 32 | 32 | 128 | MHA |
| LLaMA-2 70B | 70B | 80 | 8 | 128 | GQA |
| LLaMA-3 8B | 8B | 32 | 8 | 128 | GQA |
| LLaMA-3 70B | 70B | 80 | 8 | 128 | GQA |
| Mistral 7B | 7B | 32 | 8 | 128 | GQA |

In [ ]:
# ============================================================
# KV Cache 显存计算器
# 公式: 2 * n_layers * n_kv_heads * head_dim * seq_len * batch * dtype_bytes
# ============================================================

def calc_kv_cache_mb(n_layers: int, n_kv_heads: int, head_dim: int,
                     seq_len: int, batch_size: int = 1,
                     dtype_bytes: int = 2) -> float:
    """计算 KV cache 显存（MB）"""
    # 2 = K + V
    total_elements = 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size
    total_bytes = total_elements * dtype_bytes
    return total_bytes / (1024 ** 2)  # 转换为 MB


# 典型模型配置
models = {
    'LLaMA-7B  (MHA)':  {'n_layers': 32, 'n_kv_heads': 32, 'head_dim': 128},
    'LLaMA-2 7B (MHA)': {'n_layers': 32, 'n_kv_heads': 32, 'head_dim': 128},
    'LLaMA-3 8B (GQA)': {'n_layers': 32, 'n_kv_heads':  8, 'head_dim': 128},
    'LLaMA-2 70B(GQA)': {'n_layers': 80, 'n_kv_heads':  8, 'head_dim': 128},
    'Mistral 7B (GQA)': {'n_layers': 32, 'n_kv_heads':  8, 'head_dim': 128},
}

seq_lengths = [512, 1024, 2048, 4096, 8192]

# 打印表格
print('=' * 90)
print(f'{"模型":<22}', end='')
for s in seq_lengths:
    print(f'{f"seq={s}":>12}', end='')
print()
print('=' * 90)

for name, cfg in models.items():
    print(f'{name:<22}', end='')
    for s in seq_lengths:
        mb = calc_kv_cache_mb(**cfg, seq_len=s)
        print(f'{mb:>10.1f}MB', end='')
    print()

print('\n（以上为 batch_size=1, FP16 精度）')

# ---- 可视化 ----
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图: 不同模型的 KV cache 随序列长度变化
    ax = axes[0]
    colors = [C_RED, C_ORANGE, C_GREEN, C_BLUE, C_GRAY]
    for (name, cfg), color in zip(models.items(), colors):
        mbs = [calc_kv_cache_mb(**cfg, seq_len=s) for s in seq_lengths]
        ax.plot(seq_lengths, mbs, 'o-', color=color, label=name, linewidth=2)
    ax.set_xlabel('序列长度', fontsize=12)
    ax.set_ylabel('KV Cache (MB)', fontsize=12)
    ax.set_title('KV Cache vs 序列长度 (batch=1, FP16)', fontsize=13, color=C_DARK)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 右图: MHA vs GQA vs MQA 在不同 batch size 下
    ax = axes[1]
    batch_sizes = [1, 2, 4, 8, 16, 32]
    base_cfg = {'n_layers': 32, 'head_dim': 128, 'seq_len': 4096}
    variants = {
        f'MHA (32 KV heads)': 32,
        f'GQA (8 KV heads)':  8,
        f'MQA (1 KV head)':   1,
    }
    for (label, n_kv), color in zip(variants.items(), [C_RED, C_GREEN, C_BLUE]):
        mbs = [calc_kv_cache_mb(**base_cfg, n_kv_heads=n_kv, batch_size=b) for b in batch_sizes]
        ax.plot(batch_sizes, mbs, 'o-', color=color, label=label, linewidth=2)
    ax.set_xlabel('Batch Size', fontsize=12)
    ax.set_ylabel('KV Cache (MB)', fontsize=12)
    ax.set_title('KV Cache vs Batch Size (LLaMA-7B, seq=4096)', fontsize=13, color=C_DARK)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'可视化错误: {e}')

## 5. RoPE — 旋转位置编码

### 核心思想

RoPE（Rotary Position Embedding）由苏剑林提出，核心思想是：**用旋转矩阵编码相对位置**。

对于位置 $m$ 的查询向量 $q_m$ 和位置 $n$ 的键向量 $k_n$，RoPE 使得：

$$\langle f(q, m), f(k, n) \rangle = g(q, k, m - n)$$

即注意力分数**只依赖相对位置 $m - n$**。

### 旋转矩阵

将 $d$ 维向量分成 $d/2$ 组，每组 2 维，施加旋转：

$$f(x, m) = \begin{pmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ \vdots \end{pmatrix} \otimes \begin{pmatrix} \cos m\theta_1 \\ \cos m\theta_1 \\ \cos m\theta_2 \\ \cos m\theta_2 \\ \vdots \end{pmatrix} + \begin{pmatrix} -x_2 \\ x_1 \\ -x_4 \\ x_3 \\ \vdots \end{pmatrix} \otimes \begin{pmatrix} \sin m\theta_1 \\ \sin m\theta_1 \\ \sin m\theta_2 \\ \sin m\theta_2 \\ \vdots \end{pmatrix}$$

其中 $\theta_i = 10000^{-2i/d}$（类似正弦编码的频率）。

### 复数技巧

将相邻两个维度看作一个复数 $z = x_{2i} + x_{2i+1} \cdot j$，则旋转变为：

$$f(z, m) = z \cdot e^{jm\theta_i}$$

实现时只需复数乘法，非常高效。

### 为什么 RoPE 优于学习式 PE？

| 特性 | 学习式 PE | 正弦 PE | RoPE |
|------|:---------:|:-------:|:----:|
| 相对位置信息 | 无 | 间接 | 直接 |
| 外推能力 | 差 | 一般 | 较好 |
| 参数量 | $L \times d$ | 0 | 0 |
| 实现复杂度 | 简单 | 简单 | 中等 |

In [ ]:
# ============================================================
# RoPE (Rotary Position Embedding) 实现
# 核心: 预计算 cos/sin, 然后对 Q/K 施加旋转
# ============================================================

class RotaryPositionalEmbedding:
    """
    RoPE 实现
    1. 预计算频率 theta_i = base^(-2i/d)
    2. 预计算 cos(m * theta_i) 和 sin(m * theta_i)
    3. 应用旋转: 将向量视为复数，乘以 e^(j*m*theta)
    """
    def __init__(self, head_dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        
        # theta_i = base^(-2i/d), i = 0, 1, ..., d/2 - 1
        # 频率从高到低（捕捉从局部到全局的位置信息）
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        # inv_freq: (d/2,)
        
        # 预计算所有位置的 cos 和 sin
        positions = torch.arange(max_seq_len).float()  # (L,)
        # angles[m, i] = m * theta_i
        angles = torch.outer(positions, inv_freq)  # (L, d/2)
        
        # 缓存 cos 和 sin
        self.cos_cached = angles.cos()  # (L, d/2)
        self.sin_cached = angles.sin()  # (L, d/2)
    
    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """
        将向量的每对相邻维度交换并取反:
        [x1, x2, x3, x4, ...] -> [-x2, x1, -x4, x3, ...]
        这对应复数乘法中的 '乘以 j' 操作
        """
        x1 = x[..., :x.shape[-1] // 2]   # 前半
        x2 = x[..., x.shape[-1] // 2:]   # 后半
        return torch.cat([-x2, x1], dim=-1)
    
    def apply_rotary(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
        """
        对输入张量施加 RoPE
        x: (B, H, L, d) - Q 或 K 张量
        返回: 旋转后的张量，形状不变
        
        公式: x_rotated = x * cos + rotate_half(x) * sin
        （这等价于复数乘法 x * e^(j*theta)）
        """
        cos = self.cos_cached[:seq_len].to(x.device)  # (L, d/2)
        sin = self.sin_cached[:seq_len].to(x.device)  # (L, d/2)
        
        # 扩展维度以广播: (L, d/2) -> (1, 1, L, d/2)
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        
        # 拼接 cos/sin 到完整维度: (1, 1, L, d/2) -> (1, 1, L, d)
        cos = torch.cat([cos, cos], dim=-1)
        sin = torch.cat([sin, sin], dim=-1)
        
        # RoPE 公式
        return x * cos + self._rotate_half(x) * sin


# ---- 测试 RoPE ----
try:
    head_dim = 64
    seq_len  = 16
    rope = RotaryPositionalEmbedding(head_dim=head_dim, max_seq_len=seq_len)
    
    # 模拟 Q, K
    q = torch.randn(1, 1, seq_len, head_dim)  # (B=1, H=1, L, d)
    k = torch.randn(1, 1, seq_len, head_dim)
    
    q_rot = rope.apply_rotary(q, seq_len)
    k_rot = rope.apply_rotary(k, seq_len)
    
    print(f'Q 原始形状: {q.shape}')
    print(f'Q 旋转后:   {q_rot.shape}')
    
    # 验证: RoPE 保持向量模长
    norm_before = q.norm(dim=-1).mean()
    norm_after  = q_rot.norm(dim=-1).mean()
    print(f'\n旋转前平均模长: {norm_before:.4f}')
    print(f'旋转后平均模长: {norm_after:.4f}')
    print(f'模长保持: {torch.allclose(norm_before, norm_after, atol=1e-5)}')
    
    # 验证: 注意力分数只依赖相对位置
    # 计算 q[m] @ k[n]^T 对所有 m, n
    attn_scores = torch.matmul(q_rot, k_rot.transpose(-2, -1)).squeeze()  # (L, L)
    
    # 检查: 相同相对距离的注意力分数应该有相似的模式
    print(f'\n注意力分数矩阵形状: {attn_scores.shape}')
    print(f'对角线 (相对位置=0) 均值: {attn_scores.diag().mean():.4f}')
    
    # 可视化 RoPE 的旋转效果
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # cos 和 sin 的频率模式
    ax = axes[0]
    cos_np = rope.cos_cached[:seq_len, :8].numpy()
    for i in range(8):
        ax.plot(range(seq_len), cos_np[:, i], alpha=0.7)
    ax.set_title('cos(m * theta_i) 前8个频率', fontsize=12, color=C_DARK)
    ax.set_xlabel('位置 m')
    ax.grid(True, alpha=0.3)
    
    # 注意力分数热力图
    ax = axes[1]
    im = ax.imshow(attn_scores.detach().numpy(), cmap='RdBu_r', aspect='auto')
    ax.set_title('RoPE 注意力分数', fontsize=12, color=C_DARK)
    ax.set_xlabel('Key 位置')
    ax.set_ylabel('Query 位置')
    plt.colorbar(im, ax=ax)
    
    # 沿对角线取值 (验证相对位置性)
    ax = axes[2]
    diag_values = []
    for offset in range(-seq_len + 1, seq_len):
        vals = []
        for m in range(seq_len):
            n = m + offset
            if 0 <= n < seq_len:
                vals.append(attn_scores[m, n].item())
        if vals:
            diag_values.append((offset, np.mean(vals)))
    offsets, means = zip(*diag_values)
    ax.plot(offsets, means, color=C_RED, linewidth=2)
    ax.set_title('注意力分数 vs 相对位置', fontsize=12, color=C_DARK)
    ax.set_xlabel('相对位置 (m - n)')
    ax.set_ylabel('平均注意力分数')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'错误: {e}')

## 6. ALiBi — Attention with Linear Biases

### 核心思想

ALiBi（Attention with Linear Biases）由 Press 等人在 2022 年提出，**完全不使用位置编码**。

相反，它直接在注意力分数上加一个**线性衰减偏置**：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + m \cdot \text{bias}\right)V$$

其中偏置矩阵：

$$\text{bias}[i][j] = \begin{cases} -(i - j) & \text{if } i \geq j \\ -\infty & \text{if } i < j \end{cases}$$

每个头有不同的斜率 $m$，按几何级数设置：

$$m_i = 2^{-8i/h}, \quad i = 1, 2, \ldots, h$$

### ALiBi vs RoPE

| 特性 | RoPE | ALiBi |
|------|:----:|:-----:|
| 作用位置 | Q, K 向量 | 注意力分数 |
| 位置信息 | 旋转 Q/K | 加偏置 |
| 外推能力 | 好（配合 NTK-aware） | 极好（天然支持长序列） |
| 额外参数 | 无 | 无 |
| 代表模型 | LLaMA, Mistral, Qwen | BLOOM, MPT |

**ALiBi 的直觉**：距离越远的 token，注意力越弱（线性衰减），类似一个「注意力距离惩罚」。

In [ ]:
# ============================================================
# ALiBi (Attention with Linear Biases) 实现
# 核心: 在注意力分数上加线性距离偏置
# ============================================================

class ALiBi:
    """
    ALiBi 位置编码
    不修改 Q/K/V，而是直接在注意力分数上加偏置
    每个头有不同的衰减斜率 m = 2^(-8i/h)
    """
    def __init__(self, n_heads: int, max_seq_len: int = 4096):
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        
        # 计算每个头的斜率 m_i = 2^(-8i/h)
        # 几何级数: 从 2^(-8/h) 到 2^(-8)
        slopes = self._get_slopes(n_heads)
        
        # 预计算距离矩阵: bias[i][j] = -|i - j| (因果掩码)
        # 对于因果注意力: bias[i][j] = -(i - j) when i >= j, else -inf
        distance = torch.arange(max_seq_len).unsqueeze(0) - torch.arange(max_seq_len).unsqueeze(1)
        # distance[i][j] = i - j
        
        # 因果掩码: 只看过去的 token
        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        distance = distance.float()
        distance[~causal_mask] = float('-inf')  # 未来位置设为 -inf
        # 注意: distance[i][j] = i - j (i >= j 时为正数)
        # 我们要 bias = -distance, 所以近处距离小 -> 惩罚小
        
        # 存储偏置: (n_heads, L, L)
        # 每个头的偏置 = slope * (-distance)
        slopes_tensor = torch.tensor(slopes).unsqueeze(1).unsqueeze(2)  # (H, 1, 1)
        self.bias_cached = slopes_tensor * (-distance).unsqueeze(0)  # (H, L, L)
        # 对于因果掩码位置 (-inf), slope * (-inf) 仍为 -inf
    
    @staticmethod
    def _get_slopes(n_heads: int):
        """
        计算 ALiBi 斜率
        对于 2 的幂次个头: m = [2^(-8/h), 2^(-16/h), ..., 2^(-8)]
        对于非 2 的幂次: 插值处理
        """
        def _get_slopes_power_of_2(n):
            start = 2 ** (-8 / n)
            ratio = start
            return [start * (ratio ** i) for i in range(n)]
        
        if math.log2(n_heads).is_integer():
            return _get_slopes_power_of_2(n_heads)
        else:
            # 非 2 的幂次: 找最近的 2 的幂次，取子集
            closest_power = 2 ** math.floor(math.log2(n_heads))
            slopes = _get_slopes_power_of_2(closest_power)
            # 补充额外的斜率
            extra = 2 ** (-8 / (closest_power * 2))
            slopes_extra = [extra * (extra ** i) for i in range(n_heads - closest_power)]
            return slopes + slopes_extra
    
    def get_bias(self, seq_len: int) -> torch.Tensor:
        """获取当前序列长度的 ALiBi 偏置 (H, L, L)"""
        return self.bias_cached[:, :seq_len, :seq_len]


# ---- 测试 ALiBi ----
try:
    n_heads = 8
    seq_len = 16
    alibi = ALiBi(n_heads=n_heads, max_seq_len=seq_len)
    
    print(f'ALiBi 斜率 (每个头):')
    slopes = ALiBi._get_slopes(n_heads)
    for i, s in enumerate(slopes):
        print(f'  头 {i}: m = 2^({-8*(i+1)}/{n_heads}) = {s:.6f}')
    
    bias = alibi.get_bias(seq_len)
    print(f'\n偏置矩阵形状: {bias.shape}')
    
    # 验证: 头 0 的斜率最大（衰减最快），最后一头斜率最小（衰减最慢）
    print(f'\n头 0 (最快衰减) 偏置:')
    print(bias[0, :5, :5].numpy().round(2))
    print(f'\n头 {n_heads-1} (最慢衰减) 偏置:')
    print(bias[-1, :5, :5].numpy().round(2))
    
    # 可视化
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # 不同头的偏置热力图
    for idx, head_idx in enumerate([0, n_heads // 2, n_heads - 1]):
        ax = axes[idx]
        im = ax.imshow(bias[head_idx].numpy(), cmap='Blues_r', aspect='auto')
        ax.set_title(f'ALiBi 偏置 - 头 {head_idx} (m={slopes[head_idx]:.4f})', fontsize=11, color=C_DARK)
        ax.set_xlabel('Key 位置')
        ax.set_ylabel('Query 位置')
        plt.colorbar(im, ax=ax)
    
    plt.tight_layout()
    plt.show()
    
    # 对比: 不同头对同一 query 位置的注意力衰减
    fig, ax = plt.subplots(figsize=(8, 4))
    query_pos = seq_len - 1  # 最后一个位置
    for head_idx in range(n_heads):
        attn_decay = bias[head_idx, query_pos, :seq_len].numpy()
        ax.plot(range(seq_len), attn_decay, label=f'头{head_idx} (m={slopes[head_idx]:.4f})', alpha=0.8)
    ax.set_title(f'ALiBi: query@位置{query_pos} 对各 key 的注意力偏置', fontsize=12, color=C_DARK)
    ax.set_xlabel('Key 位置')
    ax.set_ylabel('偏置值')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'错误: {e}')

## 7. LLaMA 组件拆解

LLaMA 系列（Meta）是目前最重要的开源 LLM 架构。相比 GPT-2/3，LLaMA 做了以下关键改进：

### 7.1 Pre-RMSNorm（替代 Post-LayerNorm）

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot \gamma$$

- **Pre-Norm**：先归一化再做变换（训练更稳定）
- **RMSNorm**：去掉均值中心化，只用 RMS（更快，效果相当）

### 7.2 SwiGLU FFN（替代 ReLU/GELU FFN）

标准 FFN：
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

SwiGLU FFN：
$$\text{SwiGLU}(x) = (\text{Swish}(xW_1) \otimes xW_3) W_2$$

其中 $\text{Swish}(x) = x \cdot \sigma(x)$，$\otimes$ 是逐元素乘法。

- GLU（Gated Linear Unit）引入门控机制
- 比 ReLU FFN 多一个 $W_3$ 矩阵，但效果显著更好
- LLaMA 将 FFN 隐藏层维度设为 $\frac{2}{3} \times 4d$ 以保持参数量不变

### 7.3 RoPE（替代绝对位置编码）

在每一层的 Q 和 K 上施加旋转位置编码（见第 5 节）。

### 7.4 GQA（LLaMA-2 70B+）

在更大的模型中使用 Grouped-Query Attention（见第 3 节）。

### LLaMA Block 结构

```
输入 x
  |
  +---> RMSNorm ---> Multi-Head Attn (+ RoPE) ---> + (残差)
  |                                                  |
  +----------------------------------------------- ---> out1
  |
  +---> RMSNorm ---> SwiGLU FFN ---> + (残差)
  |                                   |
  +----------------------------------> out2
  |
输出 out2
```

In [ ]:
# ============================================================
# LLaMA 核心组件实现
# RMSNorm + SwiGLU FFN + LLaMA Block
# ============================================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization
    比 LayerNorm 更快（不计算均值）
    公式: x / RMS(x) * gamma, 其中 RMS(x) = sqrt(mean(x^2))
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # 可学习缩放参数 gamma
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D)
        # 计算 RMS: sqrt(mean(x^2))
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        # 归一化并缩放
        return x / rms * self.weight


class SwiGLUFFN(nn.Module):
    """
    SwiGLU Feed-Forward Network
    SwiGLU(x) = (Swish(x * W1) * (x * W3)) * W2
    
    注意: LLaMA 将隐藏层维度设为 2/3 * 4d (而非 4d)
    以保持与标准 ReLU FFN 相同的参数量
    """
    def __init__(self, d_model: int, hidden_dim: int = None):
        super().__init__()
        if hidden_dim is None:
            # LLaMA 的隐藏层维度: 2/3 * 4 * d_model, 然后取 256 的倍数
            hidden_dim = int(2 * 4 * d_model / 3)
            hidden_dim = 256 * ((hidden_dim + 255) // 256)  # 对齐到 256
        
        self.W_gate = nn.Linear(d_model, hidden_dim, bias=False)  # 门控分支
        self.W_up   = nn.Linear(d_model, hidden_dim, bias=False)  # 上投影
        self.W_down = nn.Linear(hidden_dim, d_model, bias=False)  # 下投影
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Swish(x * W_gate) * (x * W_up)
        gate = F.silu(self.W_gate(x))  # Swish = SiLU
        up   = self.W_up(x)
        # 门控: 逐元素乘法
        return self.W_down(gate * up)


class LLaMABlock(nn.Module):
    """
    单个 LLaMA Transformer Block
    结构: Pre-RMSNorm -> Attn -> residual -> Pre-RMSNorm -> FFN -> residual
    """
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int = None,
                 max_seq_len: int = 4096, dropout: float = 0.0):
        super().__init__()
        if n_kv_heads is None:
            n_kv_heads = n_heads  # 默认 MHA
        
        # Pre-RMSNorm (注意力前)
        self.attn_norm = RMSNorm(d_model)
        # GQA 注意力
        self.attn = GroupedQueryAttention(d_model, n_heads, n_kv_heads, dropout)
        
        # Pre-RMSNorm (FFN 前)
        self.ffn_norm = RMSNorm(d_model)
        # SwiGLU FFN
        self.ffn = SwiGLUFFN(d_model)
        
        # RoPE
        head_dim = d_model // n_heads
        self.rope = RotaryPositionalEmbedding(head_dim=head_dim, max_seq_len=max_seq_len)
    
    def forward(self, x, mask=None):
        """
        x: (B, L, D)
        """
        B, L, D = x.shape
        
        # ---- 注意力子层 (Pre-Norm) ----
        h = self.attn_norm(x)        # 先归一化
        # 注意: 完整实现中需要在 GQA 内部对 Q, K 施加 RoPE
        attn_out, _ = self.attn(h, mask=mask)  # 再做注意力
        x = x + attn_out              # 残差连接
        
        # ---- FFN 子层 (Pre-Norm) ----
        h = self.ffn_norm(x)          # 先归一化
        ffn_out = self.ffn(h)         # 再做 FFN
        x = x + ffn_out              # 残差连接
        
        return x


# ---- 测试 LLaMA 组件 ----
try:
    # 配置 (类似 LLaMA-3 8B 的缩小版)
    d_model  = 256
    n_heads  = 8
    n_kv_heads = 2  # GQA: 4:1
    seq_len  = 32
    
    block = LLaMABlock(d_model, n_heads, n_kv_heads, max_seq_len=seq_len)
    x = torch.randn(2, seq_len, d_model)
    
    causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
    out = block(x, mask=causal_mask)
    
    print(f'输入:  {x.shape}')
    print(f'输出:  {out.shape}')
    print(f'总参数: {sum(p.numel() for p in block.parameters()):,}')
    
    # 分解各组件参数量
    print(f'\n各组件参数量:')
    print(f'  Attention Norm: {sum(p.numel() for p in block.attn_norm.parameters()):>8,}')
    print(f'  Attention:      {sum(p.numel() for p in block.attn.parameters()):>8,}')
    print(f'  FFN Norm:       {sum(p.numel() for p in block.ffn_norm.parameters()):>8,}')
    print(f'  SwiGLU FFN:     {sum(p.numel() for p in block.ffn.parameters()):>8,}')
    
    # RMSNorm 验证
    rms = RMSNorm(d_model)
    test_input = torch.randn(1, 4, d_model)
    test_output = rms(test_input)
    rms_val = torch.sqrt(torch.mean(test_output ** 2, dim=-1))
    print(f'\nRMSNorm 验证 (输出 RMS 应接近 1): {rms_val.detach().numpy().round(4)}')
    
    # SwiGLU 可视化
    fig, ax = plt.subplots(figsize=(8, 4))
    x_vals = torch.linspace(-3, 3, 100)
    swish = x_vals * torch.sigmoid(x_vals)
    relu = F.relu(x_vals)
    gelu = F.gelu(x_vals)
    ax.plot(x_vals.numpy(), relu.numpy(), label='ReLU', color=C_GRAY, linewidth=2)
    ax.plot(x_vals.numpy(), gelu.numpy(), label='GELU', color=C_GREEN, linewidth=2)
    ax.plot(x_vals.numpy(), swish.numpy(), label='Swish (SiLU)', color=C_RED, linewidth=2)
    ax.set_title('激活函数对比', fontsize=13, color=C_DARK)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color=C_DARK, linewidth=0.5)
    ax.axvline(x=0, color=C_DARK, linewidth=0.5)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'错误: {e}')

## 8. Scaling Laws — Chinchilla 最优训练

### Kaplan Scaling Laws (OpenAI, 2020)

模型的测试损失 $L$ 可以用幂律描述：

$$L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}, \quad L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D}, \quad L(C) = \left(\frac{C_c}{C}\right)^{\alpha_C}$$

其中 $N$ 是参数量，$D$ 是数据量（token 数），$C$ 是计算量（FLOPs）。

### Chinchilla Scaling Law (DeepMind, 2022)

**核心发现**：对于给定的计算预算 $C$，最优的参数量 $N$ 和数据量 $D$ 应满足：

$$D \approx 20 \times N$$

即：**每个参数大约需要 20 个训练 token**。

### 实际影响

| 模型 | 参数量 | 训练 Token | 比例 | 是否 Chinchilla-optimal |
|------|:---:|:---:|:---:|:---:|
| GPT-3 | 175B | 300B | 1.7x | 严重 undertrained |
| Chinchilla | 70B | 1.4T | 20x | 最优 |
| LLaMA-1 7B | 7B | 1T | 143x | 过度训练（推理优化） |
| LLaMA-2 7B | 7B | 2T | 286x | 过度训练（推理优化） |
| LLaMA-3 8B | 8B | 15T | 1875x | 极度过度训练 |

### 为什么 LLaMA 过度训练？

Chinchilla 优化的是**训练 FLOPs**，但实际中**推理成本**更重要：

- 更小的模型 + 更多数据 = 推理更快更便宜
- Chinchilla-optimal 的 70B 模型推理成本远高于过度训练的 7B 模型
- **推理优化训练**：用更多数据训练小模型，降低部署成本

### FLOPs 估算

前向传播 FLOPs（近似）：

$$C_{forward} \approx 2N \times D$$

训练 FLOPs（前向 + 反向）：

$$C_{train} \approx 6N \times D$$

In [ ]:
# ============================================================
# Scaling Laws 可视化
# Chinchilla 最优训练: D_opt ≈ 20 * N
# ============================================================

try:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # ---- 左图: Scaling Law - Loss vs Parameters ----
    ax = axes[0]
    # 模拟 Kaplan Scaling Law: L(N) = (Nc / N)^alpha
    Nc = 8.8e13  # 临界参数量
    alpha_N = 0.076
    params = np.logspace(7, 12, 100)  # 10M 到 1T
    loss_N = (Nc / params) ** alpha_N
    ax.plot(params, loss_N, color=C_RED, linewidth=2.5, label='L(N) Scaling Law')
    
    # 标注实际模型
    model_params = {
        'GPT-2 (1.5B)': 1.5e9,
        'LLaMA-7B': 7e9,
        'LLaMA-13B': 13e9,
        'Chinchilla (70B)': 70e9,
        'GPT-3 (175B)': 175e9,
    }
    for name, n in model_params.items():
        l = (Nc / n) ** alpha_N
        ax.plot(n, l, 'o', markersize=8, color=C_DARK)
        ax.annotate(name, (n, l), textcoords='offset points',
                   xytext=(5, 8), fontsize=8, color=C_DARK)
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('参数量 N', fontsize=12)
    ax.set_ylabel('Test Loss', fontsize=12)
    ax.set_title('Scaling Law: Loss vs 参数量', fontsize=13, color=C_DARK)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=10)
    
    # ---- 中图: Chinchilla Optimal - N vs D ----
    ax = axes[1]
    N_range = np.logspace(8, 12, 50)  # 100M 到 1T
    D_opt = 20 * N_range  # Chinchilla: D ≈ 20N
    
    ax.fill_between(N_range, D_opt * 0.5, D_opt * 2, alpha=0.15, color=C_GREEN,
                   label='最优区域 (0.5x - 2x)')
    ax.plot(N_range, D_opt, color=C_GREEN, linewidth=2.5, label='D = 20N (Chinchilla)')
    
    # 标注实际模型
    real_models = {
        'GPT-3': (175e9, 300e9, C_RED),
        'Chinchilla': (70e9, 1.4e12, C_GREEN),
        'LLaMA-1 7B': (7e9, 1e12, C_BLUE),
        'LLaMA-2 7B': (7e9, 2e12, C_ORANGE),
        'LLaMA-3 8B': (8e9, 15e12, C_DARK),
    }
    for name, (n, d, c) in real_models.items():
        ratio = d / n
        ax.plot(n, d, '*', markersize=12, color=c, zorder=5)
        ax.annotate(f'{name}\n({ratio:.0f}x)', (n, d),
                   textcoords='offset points', xytext=(8, -5),
                   fontsize=8, color=c, fontweight='bold')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('参数量 N', fontsize=12)
    ax.set_ylabel('训练 Token 数 D', fontsize=12)
    ax.set_title('Chinchilla 最优训练: N vs D', fontsize=13, color=C_DARK)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=10, loc='upper left')
    
    # ---- 右图: 计算量 vs 模型大小 ----
    ax = axes[2]
    # FLOPs ≈ 6 * N * D
    N_vals = np.logspace(8, 12, 50)
    # Chinchilla optimal: D = 20N -> FLOPs = 6 * N * 20N = 120 * N^2
    flops_chinchilla = 120 * N_vals ** 2
    # 过度训练: D = 200N -> FLOPs = 6 * N * 200N = 1200 * N^2
    flops_overtrain = 1200 * N_vals ** 2
    
    ax.plot(N_vals, flops_chinchilla, color=C_GREEN, linewidth=2.5,
           label='Chinchilla (D=20N)')
    ax.plot(N_vals, flops_overtrain, color=C_RED, linewidth=2.5,
           linestyle='--', label='过度训练 (D=200N)')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('参数量 N', fontsize=12)
    ax.set_ylabel('训练 FLOPs', fontsize=12)
    ax.set_title('训练 FLOPs vs 参数量', fontsize=13, color=C_DARK)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # 数值表格
    print('=' * 65)
    print(f'{"模型":<18} {"参数量":>10} {"训练Token":>12} {"比例":>8} {"FLOPs":>15}')
    print('=' * 65)
    configs = [
        ('GPT-3',        175e9,  300e9),
        ('Chinchilla',    70e9, 1400e9),
        ('LLaMA-1 7B',     7e9, 1000e9),
        ('LLaMA-2 7B',     7e9, 2000e9),
        ('LLaMA-2 70B',   70e9, 2000e9),
        ('LLaMA-3 8B',     8e9, 15e12),
    ]
    for name, n, d in configs:
        ratio = d / n
        flops = 6 * n * d
        print(f'{name:<18} {n/1e9:>8.0f}B {d/1e12:>10.1f}T {ratio:>7.0f}x {flops:>14.2e}')
    print('\n注: FLOPs = 6 * N * D (前向+反向传播)')
except Exception as e:
    print(f'错误: {e}')

## 9. 面试 Q&A 总结

---

### Q1: MHA、MQA、GQA 的区别？显存占用如何计算？

**MHA**：$h$ 个 Q 头、$h$ 个 K 头、$h$ 个 V 头，每个头独立。

**MQA**：$h$ 个 Q 头，但只有 1 个 K 头和 1 个 V 头，所有 Q 头共享。KV cache 缩减到 $1/h$。

**GQA**：$h$ 个 Q 头，$g$ 个 K/V 头（$1 < g < h$），每 $h/g$ 个 Q 头共享一组 K/V。是 MHA 和 MQA 的折中。

KV cache = $2 \times n_{layers} \times n_{kv\_heads} \times d_{head} \times seq\_len \times batch \times dtype$。

---

### Q2: 为什么 MQA 能显著减少 KV cache？

自回归推理时，每生成一个 token 都需要缓存所有历史 token 的 K 和 V。MQA 将 KV 头数从 $h$ 减到 1，KV cache 直接缩减为 $1/h$。例如 LLaMA-70B 有 64 头，MQA 可节省 64 倍显存。

---

### Q3: RoPE 的旋转矩阵如何构造？为什么优于学习式 PE？

将 $d$ 维向量分为 $d/2$ 对，每对看作复数 $z = a + bi$。位置 $m$ 的编码是乘以旋转因子 $e^{jm\theta_k}$，其中 $\theta_k = 10000^{-2k/d}$。实现时预计算 cos/sin，用 $x \cos + \text{rotate\_half}(x) \sin$ 完成。

优势：(1) 天然编码**相对位置**（注意力分数只依赖 $m-n$）；(2) 零额外参数；(3) 配合 NTK-aware 插值可外推到更长序列。

---

### Q4: ALiBi 和 RoPE 的本质区别？

**RoPE** 作用于 Q、K 向量（旋转），**ALiBi** 作用于注意力分数矩阵（加偏置）。RoPE 通过旋转编码位置信息，ALiBi 直接施加距离衰减 $m \cdot |i-j|$。ALiBi 外推性更好（线性偏置天然适应长序列），RoPE 表达能力更强。LLaMA/Mistral 用 RoPE，BLOOM/MPT 用 ALiBi。

---

### Q5: LLaMA 相比 GPT 的架构改进？

1. **Pre-RMSNorm**（替代 Post-LayerNorm）：训练更稳定
2. **SwiGLU FFN**（替代 ReLU FFN）：门控机制，效果更好
3. **RoPE**（替代绝对位置编码）：相对位置信息，外推性好
4. **GQA**（LLaMA-2 70B+）：平衡推理效率和质量
5. **无 bias**：去掉所有线性层的 bias，简化且效果不变

---

### Q6: SwiGLU 的优势？

SwiGLU = Swish 激活 + GLU 门控。门控机制让网络可以选择性地传递信息，比 ReLU/GELU 的无门控 FFN 表达能力更强。实验表明 SwiGLU 在相同参数量下 perplexity 更低。代价是多一个 $W_3$ 矩阵，LLaMA 通过缩小隐藏层维度（$\frac{2}{3} \times 4d$ 而非 $4d$）来保持总参数量不变。

---

### Q7: Chinchilla Scaling Law 的核心结论？

对于给定计算预算，**最优的训练 token 数应约为参数量的 20 倍**（$D_{opt} \approx 20N$）。GPT-3 (175B 参数只训了 300B token) 严重 undertrained。但这只优化了训练成本；实践中为了降低**推理成本**，通常会用更多数据训练更小的模型（如 LLaMA-3 8B 用了 15T token，比例 1875x）。

---

### Q8: 如何计算 7B 模型在 seq=4096 的 KV cache？

以 LLaMA-2 7B 为例：$n_{layers}=32$，$n_{kv\_heads}=32$（MHA），$d_{head}=128$，FP16（2 字节）。

$$\text{KV cache} = 2 \times 32 \times 32 \times 128 \times 4096 \times 2 = 2.15\text{GB}$$

若使用 GQA（$n_{kv\_heads}=8$，如 LLaMA-3 8B）：$2 \times 32 \times 8 \times 128 \times 4096 \times 2 = 0.54\text{GB}$，缩减 4 倍。

In [ ]:
# ============================================================
# 课程总结
# ============================================================

summary = """
╔══════════════════════════════════════════════════════════════════╗
║          W3 Day4: LLM架构深入 - 学习总结                        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  1. MHA/MQA/GQA 注意力变体                                     ║
║     - MHA: 标准多头, h个Q/K/V头                                ║
║     - MQA: 共享K/V (1个头), KV cache 缩减 h倍                  ║
║     - GQA: 分组K/V (g个头), MHA与MQA的折中                     ║
║     - KV cache 公式: 2 * layers * kv_heads * d * seq * dtype    ║
║                                                                ║
║  2. 位置编码                                                    ║
║     - RoPE: 旋转矩阵编码相对位置, 作用于Q/K向量                 ║
║       复数技巧: z * e^(j*m*theta)                              ║
║     - ALiBi: 线性距离偏置, 作用于注意力分数                     ║
║       bias[i][j] = m * (-|i-j|)                                ║
║                                                                ║
║  3. LLaMA 架构                                                  ║
║     - Pre-RMSNorm (替代 Post-LayerNorm)                        ║
║     - SwiGLU FFN (门控机制, Swish + GLU)                       ║
║     - RoPE 位置编码                                             ║
║     - GQA (LLaMA-2 70B+)                                       ║
║     - 无 bias 线性层                                            ║
║                                                                ║
║  4. Scaling Laws                                                ║
║     - Chinchilla: D_opt ≈ 20 * N                               ║
║     - 训练 FLOPs ≈ 6 * N * D                                   ║
║     - 推理优化: 小模型 + 更多数据                               ║
║                                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  关键公式:                                                      ║
║  Attention = softmax(QK^T / sqrt(d_k)) V                       ║
║  RoPE: x*cos + rotate_half(x)*sin                              ║
║  ALiBi: scores + m * (-distance)                               ║
║  RMSNorm: x / RMS(x) * gamma                                   ║
║  SwiGLU: W_down(Swish(x*W_gate) * (x*W_up))                   ║
║  KV Cache: 2 * L * H_kv * d * seq * batch * dtype              ║
║  Chinchilla: D = 20N, FLOPs = 6ND                              ║
╚══════════════════════════════════════════════════════════════════╝
"""

print(summary)

# 快速验证所有实现
print('快速验证:')
try:
    d, h, kv_h, L = 128, 8, 2, 16
    x = torch.randn(1, L, d)
    mask = torch.tril(torch.ones(L, L)).unsqueeze(0).unsqueeze(0)
    
    # MHA
    mha = MultiHeadAttention(d, h)
    o1, _ = mha(x, mask)
    
    # MQA
    mqa = MultiQueryAttention(d, h)
    o2, _ = mqa(x, mask)
    
    # GQA
    gqa = GroupedQueryAttention(d, h, kv_h)
    o3, _ = gqa(x, mask)
    
    # LLaMA Block
    block = LLaMABlock(d, h, kv_h)
    o4 = block(x, mask)
    
    # RoPE
    rope = RotaryPositionalEmbedding(d // h, L)
    q = torch.randn(1, h, L, d // h)
    q_rot = rope.apply_rotary(q, L)
    
    # ALiBi
    alibi = ALiBi(h, L)
    bias = alibi.get_bias(L)
    
    # KV Cache 计算
    kv_mb = calc_kv_cache_mb(32, 8, 128, 4096)
    
    print(f'  MHA:        {o1.shape} ✓')
    print(f'  MQA:        {o2.shape} ✓')
    print(f'  GQA:        {o3.shape} ✓')
    print(f'  LLaMA Block:{o4.shape} ✓')
    print(f'  RoPE:       {q_rot.shape} ✓')
    print(f'  ALiBi:      {bias.shape} ✓')
    print(f'  KV Cache:   {kv_mb:.1f} MB ✓')
    print(f'\n所有组件验证通过！')
except Exception as e:
    print(f'验证错误: {e}')